<a href="https://colab.research.google.com/github/slcnvly/CQL-Reproduction/blob/master/CQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gymnasium[mujoco] wandb h5py requests

In [ ]:
%%writefile train.py
import os
import random
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import h5py
import urllib.request
import wandb

# 1. 실행 인자 세팅 (argparse)
parser = argparse.ArgumentParser()
parser.add_argument("--algo", type=str, default="cql", choices=["sac", "cql"], help="Algorithm to run")
parser.add_argument("--env", type=str, default="HalfCheetah-v4")
parser.add_argument("--seed", type=int, default=42)
args = parser.parse_args()

# 알고리즘에 따른 Alpha 값 설정 (SAC는 0, CQL은 5.0)
ALPHA_CQL = 5.0 if args.algo == "cql" else 0.0

DATASET_URL = "http://rail.eecs.berkeley.edu/datasets/offline_rl/gym_mujoco_v2/halfcheetah_medium-v2.hdf5"
BATCH_SIZE = 256
LR = 3e-4
GAMMA = 0.99
TAU = 0.005
NUM_EPOCHS = 100
STEPS_PER_EPOCH = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed(args.seed)

# 2. 데이터셋 로드
def get_d4rl_dataset():
    filepath = "halfcheetah_medium.hdf5"
    if not os.path.exists(filepath):
        urllib.request.urlretrieve(DATASET_URL, filepath)

    dataset = {}
    with h5py.File(filepath, 'r') as f:
        dataset['observations'] = f['observations'][:]
        dataset['actions'] = f['actions'][:]
        dataset['rewards'] = f['rewards'][:]
        dataset['terminals'] = f['terminals'][:]
        dataset['next_observations'] = f['next_observations'][:]
    return dataset

class OfflineBuffer:
    def __init__(self, dataset):
        self.states = torch.FloatTensor(dataset['observations']).to(device)
        self.actions = torch.FloatTensor(dataset['actions']).to(device)
        self.rewards = torch.FloatTensor(dataset['rewards']).unsqueeze(1).to(device)
        self.next_states = torch.FloatTensor(dataset['next_observations']).to(device)
        self.terminals = torch.FloatTensor(dataset['terminals']).unsqueeze(1).to(device)
        self.size = len(self.states)

    def sample(self, batch_size):
        idx = torch.randint(0, self.size, (batch_size,))
        return (self.states[idx], self.actions[idx], self.rewards[idx],
                self.next_states[idx], self.terminals[idx])

# 3. 모델 정의
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.q1 = nn.Sequential(nn.Linear(state_dim + action_dim, 256), nn.ReLU(),
                                nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1))
        self.q2 = nn.Sequential(nn.Linear(state_dim + action_dim, 256), nn.ReLU(),
                                nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1))
    def forward(self, state, action):
        sa = torch.cat([state, action], dim=-1)
        return self.q1(sa), self.q2(sa)

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 256), nn.ReLU(),
                                 nn.Linear(256, 256), nn.ReLU())
        self.mean_linear = nn.Linear(256, action_dim)
        self.log_std_linear = nn.Linear(256, action_dim)

    def sample(self, state, num_samples=1):
        x = self.net(state)
        mean = self.mean_linear(x)
        log_std = self.log_std_linear(x).clamp(-20, 2)
        std = log_std.exp()
        if num_samples > 1:
            mean = mean.unsqueeze(1).repeat(1, num_samples, 1)
            std = std.unsqueeze(1).repeat(1, num_samples, 1)
        normal = torch.distributions.Normal(mean, std)
        x_t = normal.rsample()
        action = torch.tanh(x_t)
        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(-1, keepdim=True)
        return action, log_prob

# 4. 학습 로직 (SAC / CQL 통합)
def train_step(buffer, actor, q_net, target_q_net, q_opt, actor_opt):
    state, action, reward, next_state, done = buffer.sample(BATCH_SIZE)

    with torch.no_grad():
        next_action, next_log_prob = actor.sample(next_state)
        t_q1, t_q2 = target_q_net(next_state, next_action)
        t_q = torch.min(t_q1, t_q2) - 0.2 * next_log_prob
        q_target = reward + GAMMA * (1 - done) * t_q

    q1_pred, q2_pred = q_net(state, action)
    mse_loss1 = F.mse_loss(q1_pred, q_target)
    mse_loss2 = F.mse_loss(q2_pred, q_target)

    # [CQL 페널티 계산 - algo가 'sac'이면 ALPHA_CQL이 0이 되어 무시됨]
    rand_actions = torch.empty((BATCH_SIZE, 10, action.shape[-1])).uniform_(-1, 1).to(device)
    curr_actions, _ = actor.sample(state, num_samples=10)

    state_rep = state.unsqueeze(1).repeat(1, 10, 1)
    q1_rand, _ = q_net(state_rep, rand_actions)
    q1_curr, _ = q_net(state_rep, curr_actions)

    cat_q1 = torch.cat([q1_rand, q1_curr], dim=1)
    cql_logsumexp1 = torch.logsumexp(cat_q1, dim=1).mean()
    cql_dataset_q1 = q1_pred.mean()

    cql_loss1 = cql_logsumexp1 - cql_dataset_q1

    # 최종 Loss (SAC일 경우 alpha=0이므로 mse_loss만 적용됨)
    q_loss = mse_loss1 + mse_loss2 + ALPHA_CQL * cql_loss1

    q_opt.zero_grad()
    q_loss.backward()
    q_opt.step()

    curr_act_single, curr_log_prob_single = actor.sample(state)
    q1_new, q2_new = q_net(state, curr_act_single)
    actor_loss = (0.2 * curr_log_prob_single - torch.min(q1_new, q2_new)).mean()

    actor_opt.zero_grad()
    actor_loss.backward()
    actor_opt.step()

    for p, target_p in zip(q_net.parameters(), target_q_net.parameters()):
        target_p.data.copy_(TAU * p.data + (1 - TAU) * target_p.data)

    return q_loss.item(), actor_loss.item(), cql_loss1.item(), q1_pred.mean().item()

def normalize_score(score):
    random_score = -280.18
    expert_score = 12135.0
    return (score - random_score) / (expert_score - random_score) * 100

# 5. 실행
if __name__ == "__main__":
    # WandB 프로젝트를 하나로 묶고, run name을 알고리즘 이름으로 설정
    run_name = f"{args.algo.upper()}-{args.env}"
    wandb.init(project="Offline-RL-Comparison", name=run_name, config=vars(args))

    dataset = get_d4rl_dataset()
    buffer = OfflineBuffer(dataset)
    env = gym.make(args.env)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    actor = Actor(state_dim, action_dim).to(device)
    q_net = QNetwork(state_dim, action_dim).to(device)
    t_q_net = QNetwork(state_dim, action_dim).to(device)
    t_q_net.load_state_dict(q_net.state_dict())

    q_opt = optim.Adam(q_net.parameters(), lr=LR)
    actor_opt = optim.Adam(actor.parameters(), lr=LR)

    print(f" {args.algo.upper()} Training start...")
    for epoch in range(NUM_EPOCHS):
        for step in range(STEPS_PER_EPOCH):
            q_loss, act_loss, cql_pen, q_val = train_step(buffer, actor, q_net, t_q_net, q_opt, actor_opt)

        wandb.log({"Loss/Q_loss": q_loss, "Loss/Actor_loss": act_loss,
                   "CQL/Penalty": cql_pen, "CQL/Q_value_Mean": q_val}, step=epoch)

        if epoch % 10 == 0:
            eval_scores = []
            for _ in range(3):
                state, _ = env.reset()
                done = False
                score = 0
                while not done:
                    with torch.no_grad():
                        action = torch.tanh(actor.mean_linear(actor.net(torch.FloatTensor(state).to(device)))).cpu().numpy()
                    state, reward, terminated, truncated, _ = env.step(action)
                    done = terminated or truncated
                    score += reward
                eval_scores.append(score)

            norm_score = normalize_score(np.mean(eval_scores))
            wandb.log({"Eval/Normalized_Score": norm_score}, step=epoch)
            print(f"Epoch: {epoch:>3} | Q-Value: {q_val:>7.2f} | Eval Score: {norm_score:>6.2f}")

    torch.save(actor.state_dict(), f"{args.algo}_actor.pth")
    print(f" {args.algo} 모델 저장 완료")
    wandb.finish()

In [ ]:
# SAC(기존 방식) 실험 실행
!python train.py --algo sac

In [ ]:
# CQL 실험 실행
!python train.py --algo cql

In [ ]:
# --- 가상 렌더링 환경 변수 주입 ---
import os
os.environ['MUJOCO_GL'] = 'egl'
# -----------------------------------------------------------------
import gymnasium as gym
import torch
import numpy as np
import imageio
from IPython.display import Video
import torch.nn as nn

# 1. 시뮬레이션할 알고리즘 선택 ("cql" 또는 "sac")
ALGO = "sac"

# 2. 모델 구조 재정의
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 256), nn.ReLU(),
                                 nn.Linear(256, 256), nn.ReLU())
        self.mean_linear = nn.Linear(256, action_dim)

    def forward(self, state):
        return torch.tanh(self.mean_linear(self.net(state)))

# 3. 환경 초기화
env = gym.make("HalfCheetah-v4", render_mode="rgb_array")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

# 4. 저장된 모델 불러오기
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
actor = Actor(state_dim, action_dim).to(device)
actor.load_state_dict(torch.load(f"{ALGO}_actor.pth", map_location=device), strict=False)
actor.eval()

# 5. 프레임 캡처 및 에피소드 실행
frames = []
state, _ = env.reset()
done = False
step = 0

print(f"{ALGO.upper()} 시뮬레이션 녹화 중...")
while not done and step < 1000:
    # 렌더링된 화면 프레임 저장
    frames.append(env.render())

    with torch.no_grad():
        state_tensor = torch.FloatTensor(state).to(device)
        action = actor(state_tensor).cpu().numpy()

    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    step += 1

env.close()

# 6. MP4 파일로 저장 후 출력
video_path = f"{ALGO}_simulation.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"녹화 완료, {video_path} 생성됨.")

# Colab 화면에 비디오 띄우기
Video(video_path, embed=True, width=600)

In [ ]:
# --- 가상 렌더링 환경 변수 주입 ---
import os
os.environ['MUJOCO_GL'] = 'egl'
# -----------------------------------------------------------------
import gymnasium as gym
import torch
import numpy as np
import imageio
from IPython.display import Video
import torch.nn as nn

# 1. 시뮬레이션할 알고리즘 선택 ("cql" 또는 "sac")
ALGO = "cql"

# 2. 모델 구조 재정의
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 256), nn.ReLU(),
                                 nn.Linear(256, 256), nn.ReLU())
        self.mean_linear = nn.Linear(256, action_dim)

    def forward(self, state):
        return torch.tanh(self.mean_linear(self.net(state)))

# 3. 환경 초기화
env = gym.make("HalfCheetah-v4", render_mode="rgb_array")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

# 4. 저장된 모델 불러오기
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
actor = Actor(state_dim, action_dim).to(device)
actor.load_state_dict(torch.load(f"{ALGO}_actor.pth", map_location=device), strict=False)
actor.eval()

# 5. 프레임 캡처 및 에피소드 실행
frames = []
state, _ = env.reset()
done = False
step = 0

print(f"{ALGO.upper()} 시뮬레이션 녹화 중...")
while not done and step < 1000:
    # 렌더링된 화면 프레임 저장
    frames.append(env.render())

    with torch.no_grad():
        state_tensor = torch.FloatTensor(state).to(device)
        action = actor(state_tensor).cpu().numpy()

    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    step += 1

env.close()

# 6. MP4 파일로 저장 후 출력
video_path = f"{ALGO}_simulation.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"녹화 완료, {video_path} 생성됨.")

# Colab 화면에 비디오 띄우기
Video(video_path, embed=True, width=600)